In [52]:
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder,StandardScaler,TargetEncoder,FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report,confusion_matrix,roc_curve

In [2]:
df = pd.read_csv('telco-churn-dataset.csv') # Load the dataset

## Necessary Preprocessing 

In [3]:
df = df.drop(columns=['customerID']) # Remove irrelevant column customerID

In [4]:
# Convert TotalCharges to numeric
df = df[df['TotalCharges'] != ' ']
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='raise')

In [15]:
# Also convert SeniorCitizen column to numeric
df['SeniorCitizen'] = pd.to_numeric(df['SeniorCitizen'],errors='raise')

In [ ]:
# Map gender and Churn before hand to avoid data leakage and jargaon
df['gender'] = df['gender'].map({'Male':1,"Female":0})
df['Churn'] = df['Churn'].map( lambda x : 1 if x == 'Yes' else 0)

In [35]:
def to_binary_map(cols):
    return df[cols].map( lambda x : 1 if x == 'Yes' else 0 )

In [ ]:
# Lets seperate columns into group so we can proceed easily
# Dropped churn as it will be handled seprately and gender need's a different mapping
# Also SeniorCitizen is numeric

In [41]:
binary_columns_map = [x for x in df.drop(columns=['Churn','gender','SeniorCitizen']).columns if df[x].nunique() == 2]

In [42]:
ternary_columns_map = [x for x in df.drop(columns=['InternetService','Contract','PaymentMethod']).columns if 5 > df[x].nunique() > 2 ]

In [43]:
drop_cols = ['MonthlyCharges','TotalCharges']

In [44]:
target_encoding_cols = ['InternetService','PaymentMethod']

In [ ]:
# And the remaining column Contract is to be handled by ordinal encoder 

In [48]:
df.head() # Just done a bit of preprocessing and not much

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,1,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,1,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,1,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,0,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [49]:
X , y = df.drop(columns=['Churn']),df['Churn']

In [50]:
X_train , X_test , y_train , y_test = train_test_split(
                                                            X,y,
                                                            test_size = 0.2,
                                                            stratify=y,
                                                            random_state = 42
)

In [53]:
# Also wrtting our map function inside the FunctionTransformer
to_binary_transformer = FunctionTransformer(to_binary_map)